## Model 2 — Random Forest

In [31]:
import sys

sys.path.append("../src")

In [46]:
from preprocessing import (
    load_and_clean_data,
    prepare_features,
    create_preprocessor
)

from evaluate import (
    evaluate_model,
    print_detailed_report
)

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

import pandas as pd
import joblib

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [33]:
df = load_and_clean_data(
    "../data/WA_Fn-UseC_-Telco-Customer-Churn.csv"
)

df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [34]:
X, y = prepare_features(df)

In [35]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [36]:
preprocessor = create_preprocessor(X_train)

In [37]:
random_forest_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

In [38]:
random_forest_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", random_forest_model)
    ]
)

In [39]:
random_forest_pipeline.fit(
    X_train,
    y_train
)

print("Random Forest training completed!")

Random Forest training completed!


In [40]:
rf_pred = random_forest_pipeline.predict(X_test)

print(rf_pred[:20])

[0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 1 1 1 0 0]


In [41]:
print(
    "Accuracy:",
    accuracy_score(y_test, rf_pred)
)

print(
    "Precision:",
    precision_score(y_test, rf_pred)
)

print(
    "Recall:",
    recall_score(y_test, rf_pred)
)

print(
    "F1:",
    f1_score(y_test, rf_pred)
)

Accuracy: 0.7874911158493249
Precision: 0.6306620209059234
Recall: 0.4839572192513369
F1: 0.5476550680786687


In [42]:
print(
    classification_report(
        y_test,
        rf_pred,
        target_names=["No Churn", "Churn"]
    )
)

              precision    recall  f1-score   support

    No Churn       0.83      0.90      0.86      1033
       Churn       0.63      0.48      0.55       374

    accuracy                           0.79      1407
   macro avg       0.73      0.69      0.70      1407
weighted avg       0.78      0.79      0.78      1407



In [43]:
rf_cm = confusion_matrix(
    y_test,
    rf_pred
)

print(rf_cm)

[[927 106]
 [193 181]]


In [44]:
rf_results = {
    "Model": "Random Forest",
    "Accuracy": accuracy_score(y_test, rf_pred),
    "Precision": precision_score(y_test, rf_pred),
    "Recall": recall_score(y_test, rf_pred),
    "F1": f1_score(y_test, rf_pred)
}

rf_results

{'Model': 'Random Forest',
 'Accuracy': 0.7874911158493249,
 'Precision': 0.6306620209059234,
 'Recall': 0.4839572192513369,
 'F1': 0.5476550680786687}

In [47]:
rf_pred = random_forest_pipeline.predict(
    X_test
)

In [48]:
rf_results = evaluate_model(
    "Random Forest",
    y_test,
    rf_pred
)

rf_results

{'Model': 'Random Forest',
 'Accuracy': 0.7874911158493249,
 'Precision': 0.6306620209059234,
 'Recall': 0.4839572192513369,
 'F1': 0.5476550680786687}

In [49]:
print_detailed_report(
    y_test,
    rf_pred
)

              precision    recall  f1-score   support

    No Churn       0.83      0.90      0.86      1033
       Churn       0.63      0.48      0.55       374

    accuracy                           0.79      1407
   macro avg       0.73      0.69      0.70      1407
weighted avg       0.78      0.79      0.78      1407

Confusion Matrix:
[[927 106]
 [193 181]]


In [50]:
joblib.dump(
    random_forest_pipeline,
    "../models/random_forest.joblib"
)

['../models/random_forest.joblib']

In [51]:
results_df = pd.read_csv(
    "../reports/model_results.csv"
)

In [52]:
results_df = results_df[
    results_df["Model"] != "Random Forest"
]

In [53]:
rf_df = pd.DataFrame([
    rf_results
])

results_df = pd.concat(
    [
        results_df,
        rf_df
    ],
    ignore_index=True
)

In [54]:
results_df.to_csv(
    "../reports/model_results.csv",
    index=False
)